# Stage 1 · dense Lego through update 500
Attach the **same private Stage 0 input dataset** and select GPU T4 ×2. Run the three code cells. This stage initializes the seed-42 main model on 100 training cameras with uniform pixel sampling. It stops at update 500 of its 50,000-update schedule. At the Stage 0 rate of 5.96 updates/s this is about **84 seconds of updates**, plus setup, loading all 100 views, diagnostics and export. Download the review ZIP from Output and return it before continuing.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, tempfile, time, zipfile
import torch

inputs = Path("/kaggle/input")
work = Path("/kaggle/working/recovery-main")
work.mkdir(parents=True, exist_ok=True)
run = work / "runs/lego/dense-vanilla/seed-42"
manifest = work / "manifests/lego.json"
review = work.parent / "lego-dense-vanilla-s42-first-stop.zip"
data_hash = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"

markers = list(inputs.rglob("RECOVERY_STAGE0.txt"))
if len(markers) == 1:
    repo = markers[0].parent
elif not markers:
    kits = list(inputs.rglob("stage0-input.zip"))
    assert len(kits) == 1, "Attach one version of the Stage 0 dataset."
    repo = Path(tempfile.mkdtemp(prefix="nerf-dense500-"))
    with zipfile.ZipFile(kits[0]) as archive:
        assert "RECOVERY_STAGE0.txt" in archive.namelist()
        assert all((repo / name).resolve().is_relative_to(repo) for name in archive.namelist())
        archive.extractall(repo)
else:
    raise ValueError("More than one Stage 0 input is attached.")

def project(*args):
    return [sys.executable, "-u", *map(str, args)]

hardware = {"torch": str(torch.__version__), "cuda": torch.version.cuda,
            "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
(work / "hardware.json").write_text(json.dumps(hardware, indent=2))
print(hardware)
assert torch.cuda.is_available(), "Select GPU T4 in Notebook settings."

In [ ]:
observed = subprocess.check_output(project("-c", "from recovery.training import implementation_hash; print(implementation_hash())"),
                                   cwd=repo, text=True).strip()
assert observed == code_hash, "Select the same code kit used for the reviewed Stage 0."
subprocess.run(project("-m", "recovery.data", "--scene", "lego", "--source", repo / "data/blender/lego",
                       "--output", manifest), cwd=repo, check=True)
frozen = json.loads(manifest.read_text())
assert frozen["data_hash"] == data_hash
assert [len(frozen["splits"][s]) for s in ("dense", "sparse", "test")] == [100, 8, 25]

In [ ]:
from IPython.display import display, FileLink
from PIL import Image

state_path = run / "checkpoint.pt"
def checkpoint_step():
    return int(torch.load(state_path, map_location="cpu", weights_only=True)["step"])

cmd = project("-m", "recovery.training", "--preset", "main", "--protocol", "dense-vanilla",
              "--seed", 42, "--manifest", manifest, "--run", run, "--device", "cuda:0",
              "--save-every", 500, "--stop-at", 500)
if state_path.is_file():
    if checkpoint_step() > 500:
        raise ValueError("The main run has progressed past this stage. Preserve the later checkpoint.")
    cmd.append("--resume")  # trainer verifies the full run settings and manifest identity

log_path = work / "dense500.log"
started = time.monotonic()
status = {"stage": "dense Lego 500", "target_step": 500, "complete": False}
try:
    with log_path.open("a") as log:
        with subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if proc.wait():
                raise RuntimeError(f"Dense Lego stopped with exit status {proc.returncode}; see dense500.log.")
    meta = json.loads((run / "run.json").read_text())
    health = json.loads((run / "health.json").read_text())
    last = json.loads((run / "training.jsonl").read_text().splitlines()[-1])
    assert meta["train_views"] == 100 and meta["protocol"] == "dense-vanilla" and meta["seed"] == 42
    assert meta["data_hash"] == data_hash and meta["training_config"]["implementation_hash"] == code_hash
    assert meta["training_config"]["preset"] == "main" and meta["training_config"]["steps"] == 50_000
    assert checkpoint_step() == health["step"] == last["step"] == 500
    preview = run / "train-000500.png"
    with Image.open(preview) as image:
        display(image.resize((576, 192), Image.Resampling.NEAREST))
    status.update(complete=True, step=500, run_id=meta["run_id"], steps_per_s=last["steps_per_s"])
    print(f"500 updates saved. Main throughput: {last['steps_per_s']:.2f} steps/s.")
finally:
    status["cell_elapsed_seconds"] = round(time.monotonic() - started, 2)
    pending = review.with_suffix(".tmp")
    with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_STORED) as archive:
        for source in [manifest, work / "hardware.json", log_path, *sorted(run.glob("*"))]:
            if source.is_file() and not source.name.startswith(".") and not source.name.endswith(".tmp"):
                archive.write(source, source.relative_to(work))
        archive.writestr("stage1-review.json", json.dumps(status, indent=2))
    pending.replace(review)
    os.chdir(work.parent)
    print(f"Download {review.name} from Output and send it for review.")
    display(FileLink(review.name))